# Ch01 练习参考答案：Thompson Sampling

> Bernoulli 多臂老虎机 + Beta 共轭先验 + 与 UCB1 对比。

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / 'rlenvs').exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt
from utils import set_seed
from rlenvs import MultiArmedBandit

set_seed(0)


class ThompsonBernoulliAgent:
    def __init__(self, n_arms):
        self.n_arms = n_arms
        self.alpha = np.ones(n_arms)  # Beta(1,1) 先验
        self.beta = np.ones(n_arms)

    def reset(self):
        self.alpha[:] = 1; self.beta[:] = 1

    def select_action(self):
        # 从每个臂的后验采样
        theta = np.random.beta(self.alpha, self.beta)
        return int(np.argmax(theta))

    def update(self, action, reward):
        # Bernoulli 奖励是 0/1
        if reward == 1:
            self.alpha[action] += 1
        else:
            self.beta[action] += 1


class UCBAgent:
    def __init__(self, n_arms, c=2.0):
        self.n_arms = n_arms
        self.c = c
        self.Q = np.zeros(n_arms)
        self.N = np.zeros(n_arms, dtype=int)
        self.t = 0

    def reset(self):
        self.Q[:] = 0; self.N[:] = 0; self.t = 0

    def select_action(self):
        self.t += 1
        untried = np.where(self.N == 0)[0]
        if len(untried) > 0:
            return int(untried[0])
        ucb = self.Q + self.c * np.sqrt(np.log(self.t) / self.N)
        return int(np.argmax(ucb))

    def update(self, action, reward):
        self.N[action] += 1
        self.Q[action] += (reward - self.Q[action]) / self.N[action]


def run_compare(agent_factory, env_factory, n_steps=1000, n_seeds=200):
    n_arms = None
    all_cum_regret = np.zeros((n_seeds, n_steps))
    all_opt = np.zeros((n_seeds, n_steps), dtype=bool)
    for seed in range(n_seeds):
        env = env_factory(seed)
        agent = agent_factory(env.n_arms)
        agent.reset()
        env.reset()
        a_star = env.optimal_arm()
        cum = 0.0
        for t in range(n_steps):
            a = agent.select_action()
            r = env.pull(a)
            agent.update(a, r)
            cum += env.q_star[a_star] - env.q_star[a]
            all_cum_regret[seed, t] = cum
            all_opt[seed, t] = (a == a_star)
    return all_cum_regret, all_opt


n_seeds, n_steps = 200, 1000
methods = {
    'Thompson': lambda: ThompsonBernoulliAgent,
    'UCB1 c=2': lambda: UCBAgent,
}
results = {}
for name, fac in methods.items():
    cum, opt = run_compare(fac(),
                            lambda seed: MultiArmedBandit(n_arms=10, reward_dist='bernoulli', seed=seed),
                            n_steps=n_steps, n_seeds=n_seeds)
    results[name] = (cum, opt)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for name, (cum, opt) in results.items():
    axes[0].plot(cum.mean(0), label=name, linewidth=2)
    axes[0].fill_between(np.arange(n_steps), cum.mean(0) - cum.std(0), cum.mean(0) + cum.std(0), alpha=0.15)
    sm = np.convolve(opt.mean(0), np.ones(50)/50, mode='valid')
    axes[1].plot(sm, label=name, linewidth=2)
axes[0].set_title('Cumulative regret')
axes[0].set_xlabel('step')
axes[0].legend()
axes[1].set_title('Optimal action % (smoothed w=50)')
axes[1].set_xlabel('step')
axes[1].legend()
plt.tight_layout(); plt.show()

print(f"最终累计 regret (均值 ± std):")
for name, (cum, _) in results.items():
    print(f"  {name:<12}: {cum[:, -1].mean():.1f} ± {cum[:, -1].std():.1f}")
print("\nThompson 应该明显小于 UCB1（O(√T) vs O(ln T) 在 Bernoulli 上都好但 TS 更稳）")